# 🔵 In-Process RAG on Intel GPUs with Haystack

<img src="https://haystack.deepset.ai/images/haystack-ogimage.png" width="400" style="display:inline;">

This notebook runs a RAG pipeline on **Intel® GPUs** (Intel® Arc™ / Intel® Data Center GPU Max) **without any model server** — models are loaded directly into the Python process via PyTorch's XPU backend.

This is the simplest way to use Intel GPUs with Haystack: no Docker, no vLLM server, no ports. Haystack's device abstraction **auto-detects the XPU** and places models on it. (For a high-throughput *served* setup instead, see the companion `intel_gpu_rag.ipynb` which uses vLLM.)

We use:
- `SentenceTransformersTextEmbedder` / `SentenceTransformersDocumentEmbedder` — embeddings on XPU
- `HuggingFaceLocalChatGenerator` — LLM generation on XPU

## 1. Install PyTorch with Intel GPU (XPU) support

The one hardware-specific step. The `+xpu` wheel bundles the Intel oneAPI runtime — no separate oneAPI install needed. (On an NVIDIA machine you'd install the default CUDA wheel instead; the Haystack code below is unchanged.)

In [ ]:
# PyTorch XPU build (Intel GPUs). See https://pytorch.org/docs/stable/notes/get_start_xpu.html
! pip install torch --index-url https://download.pytorch.org/whl/xpu
# Haystack + local-model dependencies
! pip install haystack-ai "transformers[sentencepiece]" "sentence-transformers>=5.0.0" accelerate

In [2]:
# Verify the Intel GPU is visible to PyTorch
import torch
print("torch:", torch.__version__)
print("XPU available:", torch.xpu.is_available())
print("XPU device count:", torch.xpu.device_count())
for i in range(torch.xpu.device_count()):
    print(f"  [{i}] {torch.xpu.get_device_name(i)}")

torch: 2.12.1+xpu
XPU available: True
XPU device count: 4


  [0] Intel(R) Arc(TM) Pro B60 Graphics
  [1] Intel(R) Arc(TM) Pro B60 Graphics
  [2] Intel(R) Arc(TM) Pro B60 Graphics
  [3] Intel(R) Arc(TM) Pro B60 Graphics


## 2. Haystack auto-selects the Intel GPU

Haystack's `ComponentDevice` resolution picks a device with precedence **CUDA > XPU > MPS > CPU**. On an Intel-GPU machine with no NVIDIA card, it automatically selects `xpu`. You don't have to configure anything — but you can pin a specific GPU with `ComponentDevice.from_str("xpu:0")`.

In [3]:
from haystack.utils.device import _get_default_device
print("Haystack auto-selected device:", _get_default_device())

Haystack auto-selected device: xpu


## 3. Index documents (embeddings on Intel GPU)

`SentenceTransformersDocumentEmbedder` loads the embedding model onto the XPU and computes vectors locally.

In [4]:
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

documents = [
    Document(content="The Eiffel Tower is located in Paris and was completed in 1889."),
    Document(content="The Great Wall of China is over 13,000 miles long."),
    Document(content="Mount Everest is the tallest mountain, at 8,849 meters."),
    Document(content="Paris is the capital of France and home to the Louvre museum."),
]

document_store = InMemoryDocumentStore(embedding_similarity_function="cosine")

document_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2",
    # device is auto-selected (xpu); or set explicitly:
    # device=ComponentDevice.from_str("xpu:0"),
)
document_embedder.warm_up()
document_store.write_documents(document_embedder.run(documents)["documents"])
print(f"Indexed {document_store.count_documents()} documents on the Intel GPU.")

/home/sdp/yx/haystack/.venv/lib/python3.12/site-packages/ddtrace/internal/module.py:314: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  self.loader.exec_module(module)


/home/sdp/yx/haystack/haystack/core/component/component.py:301: FutureWarning: `SentenceTransformersDocumentEmbedder` will be removed from Haystack in version 3.0, as it is moving to the `sentence-transformers-haystack` package. To continue using it, install that package with `pip install sentence-transformers-haystack` and update your import to `from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersDocumentEmbedder`.
  instance = super().__call__(*args, **kwargs)
2026-07-20T06:35:19.613404Z [warning  ] The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead. lineno=340 module=sentence_transformers.util.decorators


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2998.45it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.69s/it]

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.70s/it]

Indexed 4 documents on the Intel GPU.


## 4. Build the RAG pipeline

Embedding retrieval + in-process LLM generation, both on the Intel GPU.

In [5]:
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceLocalChatGenerator
from haystack.dataclasses import ChatMessage

template = [ChatMessage.from_user(
    "Answer the question using ONLY the context below.\n\n"
    "Context:\n{% for d in documents %}- {{ d.content }}\n{% endfor %}\n"
    "Question: {{ query }}\nAnswer:"
)]

rag = Pipeline()
rag.add_component("text_embedder", SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2"))
rag.add_component("retriever", InMemoryEmbeddingRetriever(document_store=document_store, top_k=2))
rag.add_component("prompt", ChatPromptBuilder(template=template, required_variables=["query", "documents"]))
rag.add_component("llm", HuggingFaceLocalChatGenerator(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",  # skip the network task-inference call
    generation_kwargs={"max_new_tokens": 100},
))

rag.connect("text_embedder.embedding", "retriever.query_embedding")
rag.connect("retriever.documents", "prompt.documents")
rag.connect("prompt.prompt", "llm.messages")

/home/sdp/yx/haystack/haystack/core/component/component.py:301: FutureWarning: `SentenceTransformersTextEmbedder` will be removed from Haystack in version 3.0, as it is moving to the `sentence-transformers-haystack` package. To continue using it, install that package with `pip install sentence-transformers-haystack` and update your import to `from haystack_integrations.components.embedders.sentence_transformers import SentenceTransformersTextEmbedder`.
  instance = super().__call__(*args, **kwargs)
/home/sdp/yx/haystack/haystack/core/component/component.py:301: FutureWarning: `HuggingFaceLocalChatGenerator` will be removed from Haystack in version 3.0, as it is moving to the `transformers-haystack` package and being renamed to `TransformersChatGenerator`. To continue using it, install that package with `pip install transformers-haystack` and update your import to `from haystack_integrations.components.generators.transformers import TransformersChatGenerator`.
  instance = super().__cal

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt: ChatPromptBuilder
  - llm: HuggingFaceLocalChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt.documents (list[Document])
  - prompt.prompt -> llm.messages (list[ChatMessage])

## 5. Ask a question

In [6]:
question = "Where is the Eiffel Tower and when was it built?"

result = rag.run({
    "text_embedder": {"text": question},
    "prompt": {"query": question},
})
print(result["llm"]["replies"][0].text)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 3426.26it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 12.09it/s]


[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The Eiffel Tower is located in Paris and was built in 1889.


## Summary

We ran a full RAG pipeline — embeddings and LLM generation — **directly on an Intel GPU, in-process**, with no model server.

Two things made this work, and neither required changing the Haystack pipeline:
1. **`pip install torch --index-url .../whl/xpu`** — the Intel GPU PyTorch build.
2. **Automatic device selection** — Haystack's `ComponentDevice` detects and uses the XPU.

The same pipeline runs on NVIDIA GPUs (install the CUDA torch wheel) or CPU (default wheel) with zero code changes.

**When to use this vs. served (vLLM)?**
- **In-process (this notebook):** simplest setup, no server; great for development, single-user, and embedding/reranking workloads.
- **Served with vLLM (`intel_gpu_rag.ipynb`):** higher throughput, concurrent requests, multi-GPU sharding; better for production serving.

**Learn more:**
- [PyTorch XPU getting started](https://pytorch.org/docs/stable/notes/get_start_xpu.html)
- [`SentenceTransformersTextEmbedder`](https://docs.haystack.deepset.ai/docs/sentencetransformerstextembedder), [`HuggingFaceLocalChatGenerator`](https://docs.haystack.deepset.ai/docs/huggingfacelocalchatgenerator)